# Занятие 5. Лабораторная (бонусная): SVM, PCA и кластеризация

Лабораторная бонусная и необязательная: прислать ее можно до 31 декабря 2026 года.
В ней восемнадцать задач:
четырнадцать обязательных и четыре бонусные, у бонусных это написано
в заголовке. Обязательные задачи дают до 10 баллов, каждая решенная бонусная —
балл сверху, итог округляется вниз и не больше 11.

Задачи в трех блоках. В первом SVM: функция потерь, опорные векторы, ядро
руками и подбор `C` и `gamma`. Во втором PCA на цифрах MNIST: разложение
через SVD, сжатие и восстановление, PCA перед классификатором. В третьем
каждый метод кластеризации с семинара руками или с подбором параметров,
а в конце все методы сравниваются в одном сетапе: где какой хорош.
Последняя задача — картинки цифр на плоскости: PCA, t-SNE и UMAP.

**Заготовки.** В каждой заготовке указано, что должно получиться: тип
и форма в аннотации, размеры входов и выходов в комментарии перед функцией.
Как считать — в теории перед задачей и в семинаре. Имена переменных
и функций менять нельзя, проверка ищет их по именам. После каждой задачи
идет ячейка с открытыми проверками; при сдаче работа дополнительно
проверяется закрытыми тестами на других данных.

**Данные.** Цифры MNIST с OpenML, шесть тысяч из семидесяти, и наборы точек
из sklearn, как на семинаре. При первом запуске нужен интернет, дальше MNIST
лежит в кэше.

In [ ]:
import time
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.cluster import DBSCAN, AgglomerativeClustering, KMeans
from sklearn.datasets import fetch_openml, make_blobs, make_circles, make_moons
from sklearn.decomposition import PCA
from sklearn.exceptions import ConvergenceWarning
from sklearn.manifold import TSNE, trustworthiness
from sklearn.metrics import adjusted_rand_score, normalized_mutual_info_score, silhouette_score
from sklearn.metrics.pairwise import rbf_kernel
from sklearn.mixture import GaussianMixture
from sklearn.model_selection import StratifiedKFold, cross_val_score, train_test_split
from sklearn.neighbors import KNeighborsClassifier, NearestNeighbors
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
import umap

warnings.filterwarnings("ignore", category=ConvergenceWarning)

plt.rcParams.update({
    "figure.figsize": (8, 4.5),
    "figure.dpi": 110,
    "axes.grid": True,
    "grid.alpha": 0.25,
    "axes.spines.top": False,
    "axes.spines.right": False,
})

BLUE, BLACK, OCHRE, GREY = "#0072CE", "#0F1418", "#C98A3C", "#9CA3AF"
CLUSTER_COLORS = np.array(["#0072CE", "#C98A3C", "#4DAF4A", "#984EA3", "#E41A1C",
                           "#FF7F00", "#A65628", "#F781BF", "#17BECF", "#6B7280"])
SEED = 42

In [ ]:
mnist = fetch_openml("mnist_784", version=1, as_frame=False, parser="auto")
X_all = mnist.data.astype(np.float32) / 255
y_all = mnist.target.astype(int)
X_digits, _, y_digits, _ = train_test_split(X_all, y_all, train_size=6000, random_state=SEED, stratify=y_all)
Xd_train, Xd_test, yd_train, yd_test = train_test_split(X_digits, y_digits, test_size=1000, random_state=SEED, stratify=y_digits)
print("обучение:", Xd_train.shape, " тестовый сет:", Xd_test.shape)

## Блок A. Метод опорных векторов

Здесь метки классов $y_i \in \{-1, +1\}$, отступ объекта
$M_i = y_i(\langle w, x_i \rangle + b)$. Задача SVM в записи sklearn:

$$J(w, b) = \frac{1}{2}\|w\|^2 + C\sum_{i=1}^{n}\max(0, 1 - M_i) \to \min_{w, b}.$$

### Задача 1. Функция потерь SVM

Функция `svm_objective` считает $J(w, b)$ для данных весов.

In [ ]:
# w: (p,), b: число, X: (n, p), y: (n,) из -1 и +1, C > 0
# возвращает число J(w, b)
def svm_objective(w: np.ndarray, b: float, X: np.ndarray, y: np.ndarray, C: float) -> float:
    return ...

In [ ]:
# --- проверка ---
X_toy = np.array([[2.0, 0.0], [0.0, 2.0], [-1.0, -1.0]])
y_toy = np.array([1, 1, -1])
assert np.isclose(svm_objective(np.array([1.0, 1.0]), 0.0, X_toy, y_toy, C=1.0), 1.0), "все отступы не меньше 1: остается половина квадрата нормы"
assert np.isclose(svm_objective(np.array([0.5, 0.0]), 0.0, X_toy, y_toy, C=2.0), 0.125 + 2 * (0 + 1 + 0.5))

X_lin, y_lin01 = make_blobs(n_samples=200, centers=2, cluster_std=2.0, random_state=SEED)
y_lin = 2 * y_lin01 - 1
svc_lin = SVC(kernel="linear", C=0.5).fit(X_lin, y_lin)
w_opt, b_opt = svc_lin.coef_[0], svc_lin.intercept_[0]
best = svm_objective(w_opt, b_opt, X_lin, y_lin, C=0.5)
shaken = [svm_objective(w_opt + np.random.default_rng(k).normal(0, 0.05, 2), b_opt, X_lin, y_lin, C=0.5) for k in range(20)]
print(f"J в решении SVC: {best:.3f}, после случайных сдвигов весов: от {min(shaken):.3f}")
assert best <= min(shaken) + 1e-6, "в решении SVC целевая функция должна быть меньше, чем рядом с ним"
print("проверки пройдены")

### Задача 2. Опорные векторы

Опорные векторы — объекты с отступом $M_i \leq 1$: на краю полосы, внутри
нее и на чужой стороне. Функция `support_mask` возвращает булев вектор:
`True` для опорных векторов. Отступ сравнивается с $1 + \text{tol}$,
чтобы вычислительная погрешность не выкидывала объекты с края полосы.

In [ ]:
# w: (p,), b: число, X: (n, p), y: (n,) из -1 и +1
# возвращает (n,) булев вектор: True у опорных векторов
def support_mask(w: np.ndarray, b: float, X: np.ndarray, y: np.ndarray, tol: float = 1e-6) -> np.ndarray:
    return ...

In [ ]:
# --- проверка ---
mask = support_mask(w_opt, b_opt, X_lin, y_lin, tol=1e-3)
assert mask.dtype == bool and mask.shape == (len(y_lin),)
print("опорных векторов:", mask.sum(), " у SVC:", len(svc_lin.support_))
assert set(np.flatnonzero(mask)) >= set(svc_lin.support_), "все опорные векторы SVC должны попасть в маску"

refit = SVC(kernel="linear", C=0.5).fit(X_lin[mask], y_lin[mask])
print("веса на всех объектах:", np.round(w_opt, 4), " только на опорных:", np.round(refit.coef_[0], 4))
assert np.allclose(refit.coef_[0], w_opt, atol=1e-3), "SVM, обученный только на опорных векторах, проводит ту же границу"
print("проверки пройдены")

**Вопрос.** Почему граница не изменилась, когда выбросили все неопорные объекты?
Сохранится ли это свойство у логистической регрессии?

*Ответ пишите здесь.*

### Задача 3. RBF-ядро руками

Матрица ядра для двух наборов объектов: $K_{ij} = \exp(-\gamma \|a_i - b_j\|^2)$.
Функция `rbf_kernel_matrix` считает ее без циклов по объектам. Квадрат
расстояния раскрывается так: $\|a - b\|^2 = \|a\|^2 + \|b\|^2 - 2\langle a, b \rangle$.
Из-за округления разность может стать чуть меньше нуля, такие значения
надо заменить нулем.

In [ ]:
# A: (n, p), B: (m, p), gamma > 0
# возвращает (n, m): K[i, j] = exp(-gamma * ||A[i] - B[j]||^2)
def rbf_kernel_matrix(A: np.ndarray, B: np.ndarray, gamma: float) -> np.ndarray:
    return ...

In [ ]:
# --- проверка ---
X_moons, y_moons = make_moons(n_samples=300, noise=0.25, random_state=SEED)
Xm_train, Xm_test, ym_train, ym_test = train_test_split(X_moons, y_moons, test_size=0.3, random_state=SEED, stratify=y_moons)

mine = rbf_kernel_matrix(Xm_train, Xm_test, gamma=2.0)
assert mine.shape == (len(Xm_train), len(Xm_test))
assert np.allclose(mine, rbf_kernel(Xm_train, Xm_test, gamma=2.0)), "не совпало с sklearn.metrics.pairwise.rbf_kernel"
assert np.allclose(np.diag(rbf_kernel_matrix(Xm_train, Xm_train, 2.0)), 1.0), "ядро объекта с самим собой равно 1"

precomputed = SVC(kernel="precomputed", C=1).fit(rbf_kernel_matrix(Xm_train, Xm_train, 2.0), ym_train)
builtin = SVC(kernel="rbf", gamma=2.0, C=1).fit(Xm_train, ym_train)
assert np.array_equal(precomputed.predict(rbf_kernel_matrix(Xm_test, Xm_train, 2.0)), builtin.predict(Xm_test)), "SVC на своей матрице ядра должен отвечать так же, как с kernel='rbf'"
print("проверки пройдены")

### Задача 4. Подбор `C` и `gamma`

Функция `svm_cv_grid` для каждой пары `C` и `gamma` считает среднюю accuracy
кросс-валидации модели `make_pipeline(StandardScaler(), SVC(kernel="rbf", C=C, gamma=gamma))`
через `cross_val_score` с переданным разбиением `cv`. Результат — таблица:
строки — значения `C` в порядке списка, индекс называется `C`; столбцы —
значения `gamma`, их имя `gamma`.

После функции заполните `best_C` и `best_gamma` — пару с наибольшей средней
accuracy на «лунах» из задачи 3.

In [ ]:
# X: (n, p), y: (n,), Cs и gammas — списки чисел, cv — разбиение для cross_val_score
# возвращает таблицу (len(Cs), len(gammas)): индекс с именем "C", столбцы с именем "gamma",
# в клетке — средняя accuracy кросс-валидации
def svm_cv_grid(X: np.ndarray, y: np.ndarray, Cs: list[float], gammas: list[float], cv) -> pd.DataFrame:
    return ...


grid_cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
moons_grid = ...   # svm_cv_grid на Xm_train, ym_train, C из [0.1, 1, 10, 100], gamma из [0.1, 1, 10]
best_C, best_gamma = ..., ...

In [ ]:
# --- проверка ---
assert moons_grid.shape == (4, 3) and moons_grid.index.name == "C" and moons_grid.columns.name == "gamma"
assert moons_grid.to_numpy().max() == moons_grid.loc[best_C, best_gamma], "best_C и best_gamma — клетка с наибольшей accuracy"
plt.figure(figsize=(5.5, 4))
plt.imshow(moons_grid.to_numpy(dtype=float), cmap="Greens")
plt.xticks(range(3), moons_grid.columns)
plt.yticks(range(4), moons_grid.index)
plt.xlabel("gamma")
plt.ylabel("C")
plt.colorbar(label="accuracy кросс-валидации")
plt.grid(False)
plt.show()
final_svm = make_pipeline(StandardScaler(), SVC(kernel="rbf", C=best_C, gamma=best_gamma)).fit(Xm_train, ym_train)
print(f"лучшие C = {best_C}, gamma = {best_gamma}; accuracy на тестовом сете {final_svm.score(Xm_test, ym_test):.3f}")
print("проверки пройдены")

## Блок B. Метод главных компонент

### Задача 5. PCA через SVD

Функция `pca_svd` центрирует данные, делает сингулярное разложение
$X - \mu = U S V^\top$ и возвращает три объекта:

- `mean` — вектор средних $\mu$;
- `components` — первые $k$ строк $V^\top$, по одной компоненте в строке;
- `explained_variance` — дисперсии вдоль них, $\frac{s_j^2}{n - 1}$, как в sklearn.

In [ ]:
# X: (n, p), 1 <= k <= min(n, p)
# возвращает mean: (p,), components: (k, p), explained_variance: (k,) по убыванию
def pca_svd(X: np.ndarray, k: int) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    return ...

In [ ]:
# --- проверка ---
mean_d, components_d, variance_d = pca_svd(Xd_train, 20)
reference = PCA(n_components=20, svd_solver="full").fit(Xd_train)
assert components_d.shape == (20, 784) and variance_d.shape == (20,) and mean_d.shape == (784,)
assert np.allclose(variance_d, reference.explained_variance_, rtol=1e-3), "дисперсии не совпали с PCA из sklearn: делитель n - 1"
alignment = np.abs((components_d * reference.components_).sum(axis=1))
assert np.allclose(alignment, 1, atol=1e-3), "компоненты должны совпадать с sklearn с точностью до знака"
print("проверки пройдены")

### Задача 6. Сжать и восстановить

`pca_project` переводит объекты в координаты компонент: $z = (x - \mu) V_k$,
где $V_k$ — компоненты по столбцам. `pca_restore` возвращает их обратно:
$\hat{x} = \mu + z V_k^\top$.

In [ ]:
# X: (n, p), mean: (p,), components: (k, p); возвращает (n, k)
def pca_project(X: np.ndarray, mean: np.ndarray, components: np.ndarray) -> np.ndarray:
    return ...


# Z: (n, k), mean: (p,), components: (k, p); возвращает (n, p)
def pca_restore(Z: np.ndarray, mean: np.ndarray, components: np.ndarray) -> np.ndarray:
    return ...

In [ ]:
# --- проверка ---
Z_test = pca_project(Xd_test, mean_d, components_d)
assert Z_test.shape == (len(Xd_test), 20)
signs = np.sign((components_d * reference.components_).sum(axis=1))
assert np.allclose(Z_test * signs, reference.transform(Xd_test), atol=1e-3), "координаты не совпали с PCA.transform"

errors = {}
for k in [5, 20, 50, 150]:
    mean_k, components_k, _ = pca_svd(Xd_train, k)
    restored = pca_restore(pca_project(Xd_test, mean_k, components_k), mean_k, components_k)
    errors[k] = float(((Xd_test - restored) ** 2).mean())
print("среднеквадратичная ошибка восстановления:", {k: round(v, 4) for k, v in errors.items()})
assert all(a > b for a, b in zip(list(errors.values()), list(errors.values())[1:])), "с ростом k ошибка должна падать"

fig, axes = plt.subplots(1, 5, figsize=(10, 2.3))
mean_show, components_show, _ = pca_svd(Xd_train, 150)
for ax, k in zip(axes, [5, 20, 50, 150, None]):
    image = Xd_test[0] if k is None else pca_restore(pca_project(Xd_test[:1], mean_show, components_show[:k]), mean_show, components_show[:k])[0]
    ax.imshow(image.reshape(28, 28), cmap="gray_r")
    ax.set_title("исходная" if k is None else f"k = {k}", fontsize=9)
    ax.axis("off")
plt.show()
print("проверки пройдены")

### Задача 7. Сколько компонент оставить

`n_components_for_share` по долям объясненной дисперсии компонент
(отсортированы по убыванию, в сумме не больше 1) возвращает наименьшее $k$,
при котором первые $k$ компонент объясняют не меньше доли `share`.

In [ ]:
# ratios: (m,) доли дисперсии по убыванию, 0 < share <= сумма ratios
# возвращает целое k от 1 до m
def n_components_for_share(ratios: np.ndarray, share: float) -> int:
    return ...

In [ ]:
# --- проверка ---
assert n_components_for_share(np.array([0.5, 0.3, 0.2]), 0.5) == 1, "ровно 0.5 уже достаточно"
assert n_components_for_share(np.array([0.5, 0.3, 0.2]), 0.81) == 3
full_ratios = PCA().fit(Xd_train).explained_variance_ratio_
digits_k = {share: n_components_for_share(full_ratios, share) for share in [0.8, 0.9, 0.95, 0.99]}
print("компонент для доли дисперсии:", digits_k)
print("проверки пройдены")

### Задача 8. PCA перед kNN

Функция `pca_knn_table` для каждого $k$ из списка обучает
`make_pipeline(PCA(n_components=k, random_state=SEED), KNeighborsClassifier(n_neighbors=n_neighbors))`
на обучении и считает accuracy на тестовом сете. Результат — таблица
с индексом `компонент` (значения $k$ в порядке списка) и столбцами
`accuracy` и `секунд` — время предсказания на тестовом сете.

In [ ]:
# X_train: (n, p), y_train: (n,), X_test: (m, p), y_test: (m,), ks — список целых
# возвращает таблицу (len(ks), 2): индекс "компонент", столбцы "accuracy" и "секунд"
def pca_knn_table(X_train: np.ndarray, y_train: np.ndarray, X_test: np.ndarray, y_test: np.ndarray,
                  ks: list[int], n_neighbors: int = 5) -> pd.DataFrame:
    return ...

In [ ]:
# --- проверка ---
knn_table = pca_knn_table(Xd_train, yd_train, Xd_test, yd_test, [5, 10, 20, 40, 80, 160])
assert list(knn_table.index) == [5, 10, 20, 40, 80, 160] and knn_table.index.name == "компонент"
assert list(knn_table.columns) == ["accuracy", "секунд"]
print(knn_table.round(3))
fig, axes = plt.subplots(1, 2, figsize=(12, 3.5))
axes[0].plot(knn_table.index, knn_table["accuracy"], "o-", color=BLUE)
axes[0].set_ylabel("accuracy на тестовом сете")
axes[1].plot(knn_table.index, knn_table["секунд"], "o-", color=OCHRE)
axes[1].set_ylabel("время предсказания, с")
for ax in axes:
    ax.set_xscale("log")
    ax.set_xlabel("компонент")
plt.tight_layout()
plt.show()
print("проверки пройдены")

**Вопрос.** С какого числа компонент точность перестает расти? Почему
на очень большом числе компонент она может стать даже чуть ниже?

*Ответ пишите здесь.*

## Блок C. Кластеризация

Наборы точек — те же пять, что на семинаре, все стандартизованы.

In [ ]:
X_aniso, y_aniso = make_blobs(n_samples=500, centers=3, random_state=170)
X_aniso = X_aniso @ np.array([[0.6, -0.64], [-0.41, 0.85]])
X_density, y_density = make_blobs(n_samples=500, centers=3, cluster_std=[1.0, 2.5, 0.5], random_state=170)
datasets = {
    "шары": make_blobs(n_samples=500, centers=3, random_state=8),
    "вытянутые": (X_aniso, y_aniso),
    "разная плотность": (X_density, y_density),
    "луны": make_moons(n_samples=500, noise=0.06, random_state=SEED),
    "кольца": make_circles(n_samples=500, factor=0.5, noise=0.05, random_state=SEED),
}
datasets = {name: (StandardScaler().fit_transform(X), y) for name, (X, y) in datasets.items()}


def plot_partition(ax, X, labels, title):
    labels = np.asarray(labels)
    colors = np.where(labels >= 0, CLUSTER_COLORS[labels % len(CLUSTER_COLORS)], "#111111")
    ax.scatter(X[:, 0], X[:, 1], s=7, c=colors)
    ax.set_title(title, fontsize=10)
    ax.set_xticks([])
    ax.set_yticks([])

### Задача 9. KMeans руками

Функция `kmeans_lloyd` делает `n_iter` шагов алгоритма Ллойда, начиная
с центров `initial_centers`. На каждом шаге:

1. каждый объект относится к ближайшему центру, при равенстве — к центру
   с меньшим номером;
2. запоминается инерция $J = \sum_i \|x_i - \mu_{c(i)}\|^2$ при этом отнесении;
3. каждый центр становится средним своих объектов; если у центра объектов нет,
   он остается на месте.

Возвращаются номера кластеров, посчитанные по итоговым центрам, итоговые центры
и список инерций всех шагов. Исходный массив центров менять нельзя.

In [ ]:
# X: (n, p), initial_centers: (k, p), n_iter >= 1
# возвращает labels: (n,) целые от 0 до k-1, centers: (k, p), history: список из n_iter чисел
def kmeans_lloyd(X: np.ndarray, initial_centers: np.ndarray, n_iter: int) -> tuple[np.ndarray, np.ndarray, list[float]]:
    return ...

In [ ]:
# --- проверка ---
X_balls, y_balls = datasets["шары"]
start_centers = X_balls[[0, 1, 2]].copy()
my_labels, my_centers, my_history = kmeans_lloyd(X_balls, start_centers, n_iter=10)
assert np.array_equal(start_centers, X_balls[[0, 1, 2]]), "исходные центры не должны меняться"
assert len(my_history) == 10 and all(a >= b - 1e-9 for a, b in zip(my_history, my_history[1:])), "инерция не должна расти"
sk = KMeans(n_clusters=3, init=X_balls[[0, 1, 2]], n_init=1, max_iter=10, algorithm="lloyd", tol=0).fit(X_balls)
assert np.allclose(np.sort(my_centers, axis=0), np.sort(sk.cluster_centers_, axis=0), atol=1e-6), "центры не совпали с KMeans из sklearn с тем же стартом"
assert adjusted_rand_score(sk.labels_, my_labels) == 1.0
print("инерция по шагам:", np.round(my_history, 1))
print("проверки пройдены")

### Задача 10, бонус. Старт k-means++

`kmeans_plus_plus` выбирает $k$ стартовых центров среди объектов. Первый —
случайный объект, равновероятно. Каждый следующий — случайный объект
с вероятностью, пропорциональной квадрату расстояния до ближайшего из уже
выбранных центров. Случайность берется только из генератора `rng`.

In [ ]:
# X: (n, p), 1 <= k <= n, rng — np.random.Generator
# возвращает (k, p): строки — выбранные объекты X
def kmeans_plus_plus(X: np.ndarray, k: int, rng: np.random.Generator) -> np.ndarray:
    return ...

In [ ]:
# --- проверка ---
X_four, y_four = make_blobs(n_samples=400, centers=[[0, 0], [10, 0], [0, 10], [10, 10]], cluster_std=0.5, random_state=SEED)
hits = 0
for seed in range(100):
    centers = kmeans_plus_plus(X_four, 4, np.random.default_rng(seed))
    assert centers.shape == (4, 2) and all((X_four == c).all(axis=1).any() for c in centers), "центры — это объекты из X"
    owners = [y_four[(X_four == c).all(axis=1).argmax()] for c in centers]
    hits += len(set(owners)) == 4
print(f"по одному центру в каждом из четырех облаков: {hits} запусков из 100; у случайного выбора около 9")
assert hits >= 80, "k-means++ почти всегда разводит центры по разным облакам"
print("проверки пройдены")

### Задача 11. Сколько кластеров

Функция `choose_k_table` для каждого $k$ из списка обучает
`KMeans(n_clusters=k, n_init=10, random_state=SEED)` и возвращает таблицу
с индексом `k` и столбцами `инерция` и `силуэт`. Потом заполните `best_k` —
число кластеров с наибольшим силуэтом на наборе `X_five`.

In [ ]:
# X: (n, p), ks — список целых не меньше 2
# возвращает таблицу (len(ks), 2): индекс "k", столбцы "инерция" и "силуэт"
def choose_k_table(X: np.ndarray, ks: list[int]) -> pd.DataFrame:
    return ...


X_five, _ = make_blobs(n_samples=750, centers=5, cluster_std=1.1, random_state=7)
k_table = ...   # choose_k_table на X_five для k от 2 до 10
best_k = ...

In [ ]:
# --- проверка ---
assert list(k_table.index) == list(range(2, 11)) and list(k_table.columns) == ["инерция", "силуэт"]
fig, axes = plt.subplots(1, 2, figsize=(12, 3.5))
axes[0].plot(k_table.index, k_table["инерция"], "o-", color=BLUE)
axes[0].set_ylabel("инерция")
axes[1].plot(k_table.index, k_table["силуэт"], "o-", color=OCHRE)
axes[1].set_ylabel("силуэт")
for ax in axes:
    ax.set_xlabel("k")
plt.tight_layout()
plt.show()
print("лучшее k по силуэту:", best_k)
assert best_k == k_table["силуэт"].idxmax()
print("проверки пройдены")

### Задача 12, бонус. Силуэт руками

`silhouette_by_hand` считает средний силуэт по формуле с семинара:
$a(i)$ — среднее расстояние до остальных объектов своего кластера,
$b(i)$ — наименьшее по чужим кластерам среднее расстояние до их объектов,
$s(i) = \frac{b(i) - a(i)}{\max(a(i), b(i))}$. У объекта, который один
в своем кластере, $s(i) = 0$. Кластеров не меньше двух.

In [ ]:
# X: (n, p), labels: (n,) номера кластеров, кластеров не меньше двух
# возвращает число от -1 до 1
def silhouette_by_hand(X: np.ndarray, labels: np.ndarray) -> float:
    return ...

In [ ]:
# --- проверка ---
some_labels = KMeans(n_clusters=4, n_init=10, random_state=SEED).fit_predict(X_balls)
assert np.isclose(silhouette_by_hand(X_balls, some_labels), silhouette_score(X_balls, some_labels)), "не совпало с silhouette_score"
single = np.array([0, 0, 0, 1])
X_single = np.array([[0.0], [1.0], [2.0], [10.0]])
assert np.isclose(silhouette_by_hand(X_single, single), silhouette_score(X_single, single)), "объект один в кластере получает 0"
print("проверки пройдены")

### Задача 13. Радиус для DBSCAN

`k_distance` для каждого объекта находит расстояние до его $k$-го ближайшего
соседа, считая сам объект первым, то есть `NearestNeighbors(n_neighbors=k)`
и последний столбец расстояний, и возвращает эти расстояния по возрастанию.

Постройте график для «лун» с `min_samples = 5` и по излому выберите
`moons_eps`: DBSCAN с таким радиусом и `min_samples = 5` должен разделить
луны с ARI не меньше 0.95.

In [ ]:
# X: (n, p), k >= 1
# возвращает (n,) расстояния до k-го соседа (сам объект — первый) по возрастанию
def k_distance(X: np.ndarray, k: int) -> np.ndarray:
    return ...


moons_eps = ...

In [ ]:
# --- проверка ---
X_moon_set, y_moon_set = datasets["луны"]
kdist = k_distance(X_moon_set, 5)
assert kdist.shape == (len(X_moon_set),) and np.all(np.diff(kdist) >= 0)
assert np.allclose(k_distance(np.array([[0.0], [1.0], [3.0]]), 2), [1.0, 1.0, 2.0])
plt.figure(figsize=(8, 3.5))
plt.plot(kdist, color=BLUE)
plt.axhline(moons_eps, color=OCHRE, ls="--", label=f"moons_eps = {moons_eps}")
plt.ylabel("расстояние до 5-го соседа")
plt.legend()
plt.show()
moons_labels = DBSCAN(eps=moons_eps, min_samples=5).fit_predict(X_moon_set)
print(f"ARI {adjusted_rand_score(y_moon_set, moons_labels):.3f}, шум {np.mean(moons_labels == -1):.1%}")
assert adjusted_rand_score(y_moon_set, moons_labels) >= 0.95
print("проверки пройдены")

### Задача 14, бонус. DBSCAN руками

`dbscan_by_hand` повторяет DBSCAN: ядровые точки — у которых в радиусе `eps`,
считая саму точку, не меньше `min_samples` точек; кластер — ядровые точки,
связанные цепочками соседей, плюс неядровые точки в радиусе `eps` от его ядровых;
остальное — шум с меткой $-1$. Кластеры нумеруются с нуля. Если неядровая точка
рядом с ядровыми точками двух кластеров, ее можно отнести к любому из них.

In [ ]:
# X: (n, p), eps > 0, min_samples >= 1
# возвращает (n,) целые: номера кластеров с нуля, шум — -1
def dbscan_by_hand(X: np.ndarray, eps: float, min_samples: int) -> np.ndarray:
    return ...

In [ ]:
# --- проверка ---
for name in ["луны", "разная плотность"]:
    X, _ = datasets[name]
    mine = dbscan_by_hand(X, 0.2, 5)
    sk = DBSCAN(eps=0.2, min_samples=5).fit(X)
    core = np.zeros(len(X), dtype=bool)
    core[sk.core_sample_indices_] = True
    assert np.array_equal(mine == -1, sk.labels_ == -1), f"{name}: шум не совпал с DBSCAN из sklearn"
    assert adjusted_rand_score(sk.labels_[core], mine[core]) == 1.0, f"{name}: ядровые точки разбиты на кластеры не так, как в sklearn"
print("проверки пройдены")

### Задача 15. Число компонент смеси по BIC

`gmm_bic` для каждого числа компонент из списка обучает
`GaussianMixture(n_components=k, random_state=SEED)` и возвращает `pd.Series`
с индексом `k` и значениями BIC. Потом заполните `best_components` — число
компонент с наименьшим BIC на `X_five` из задачи 11.

In [ ]:
# X: (n, p), ks — список целых не меньше 1
# возвращает pd.Series длины len(ks): индекс "k", значения — BIC
def gmm_bic(X: np.ndarray, ks: list[int]) -> pd.Series:
    return ...


bic_five = ...   # gmm_bic на X_five для k от 1 до 10
best_components = ...

In [ ]:
# --- проверка ---
assert list(bic_five.index) == list(range(1, 11)) and bic_five.index.name == "k"
plt.figure(figsize=(8, 3.5))
plt.plot(bic_five.index, bic_five.to_numpy(), "o-", color=BLUE)
plt.xlabel("число компонент")
plt.ylabel("BIC")
plt.show()
print("лучшее число компонент по BIC:", best_components, " по силуэту KMeans:", best_k)
assert best_components == bic_five.idxmin()
print("проверки пройдены")

### Задача 16. Все методы в одном сетапе

Функция `evaluate_clustering` принимает словарь наборов `{имя: (X, y)}`
и словарь методов `{имя: функция}`, где функция по числу кластеров $k$
возвращает новую модель с методом `fit_predict`. Для каждого набора
и каждого метода модель создается с $k$, равным числу настоящих классов
набора, и обучается через `fit_predict(X)`. Результат — таблица, по строке
на пару «набор, метод» в порядке словарей, столбцы:

- `набор`, `метод`;
- `ARI`, `NMI` — против настоящих меток;
- `силуэт` — `silhouette_score(X, labels)`, шум считается отдельной меткой;
  если различных меток меньше двух, `np.nan`;
- `кластеров` — сколько кластеров найдено, без шума.

In [ ]:
# datasets: {имя: (X (n, p), y (n,))}, methods: {имя: функция k -> модель с fit_predict}
# возвращает таблицу (len(datasets) * len(methods), 6),
# столбцы "набор", "метод", "ARI", "NMI", "силуэт", "кластеров"
def evaluate_clustering(datasets: dict[str, tuple[np.ndarray, np.ndarray]], methods: dict) -> pd.DataFrame:
    return ...


methods = {
    "KMeans": lambda k: KMeans(n_clusters=k, n_init=10, random_state=SEED),
    "агломеративная, ward": lambda k: AgglomerativeClustering(n_clusters=k, linkage="ward"),
    "агломеративная, single": lambda k: AgglomerativeClustering(n_clusters=k, linkage="single"),
    "DBSCAN": lambda k: DBSCAN(eps=0.2, min_samples=5),
    "смесь гауссиан": lambda k: GaussianMixture(n_components=k, random_state=SEED),
}
clustering_table = ...   # evaluate_clustering на datasets и methods

In [ ]:
# --- проверка ---
assert list(clustering_table.columns) == ["набор", "метод", "ARI", "NMI", "силуэт", "кластеров"]
assert len(clustering_table) == len(datasets) * len(methods)
ari_pivot = clustering_table.pivot(index="набор", columns="метод", values="ARI")
plt.figure(figsize=(9, 3.8))
plt.imshow(ari_pivot.to_numpy(), cmap="Greens", vmin=0, vmax=1, aspect="auto")
for i in range(ari_pivot.shape[0]):
    for j in range(ari_pivot.shape[1]):
        plt.text(j, i, f"{ari_pivot.iloc[i, j]:.2f}", ha="center", va="center", fontsize=9)
plt.xticks(range(ari_pivot.shape[1]), ari_pivot.columns, rotation=20)
plt.yticks(range(ari_pivot.shape[0]), ari_pivot.index)
plt.title("ARI: наборы и методы")
plt.grid(False)
plt.show()
print("проверки пройдены")

**Вопрос.** По таблице и картинкам ответьте для каждого метода: на каких
наборах он хорош и почему, где ломается и почему. Совпадает ли выбор лучшего
метода по ARI и по силуэту? Почему?

*Ответ пишите здесь.*

### Задача 17, бонус. KMeans на цифрах

`digits_kmeans_by_pca` для каждого $k$ из списка сжимает цифры
`PCA(n_components=k, random_state=SEED)` и кластеризует
`KMeans(n_clusters=10, n_init=10, random_state=SEED)`. Результат — таблица
с индексом `компонент` и столбцами `ARI` и `NMI` против настоящих цифр.

In [ ]:
# X: (n, p), y: (n,), ks — список целых
# возвращает таблицу (len(ks), 2): индекс "компонент", столбцы "ARI" и "NMI"
def digits_kmeans_by_pca(X: np.ndarray, y: np.ndarray, ks: list[int]) -> pd.DataFrame:
    return ...

In [ ]:
# --- проверка ---
digits_table = digits_kmeans_by_pca(Xd_train[:2000], yd_train[:2000], [2, 5, 10, 30, 100])
assert list(digits_table.index) == [2, 5, 10, 30, 100] and list(digits_table.columns) == ["ARI", "NMI"]
print(digits_table.round(3))
print("проверки пройдены")

**Вопрос.** Почему на двух компонентах цифры кластеризуются плохо, а после
десятка компонент качество почти не растет?

*Ответ пишите здесь.*

### Задача 18. Цифры на плоскости

Функция `embedding_scores` сравнивает двумерные картинки одних и тех же
объектов. Для каждой картинки из словаря `embeddings` она считает:

- `trustworthiness` — `trustworthiness(X, points, n_neighbors=n_neighbors)`,
  насколько соседи на плоскости — настоящие соседи в `X`;
- `accuracy kNN` — средняя accuracy `KNeighborsClassifier(n_neighbors=5)` на точках
  картинки через `cross_val_score` с разбиением
  `StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)`.

Результат — таблица: строки — имена картинок в порядке словаря, два столбца.

Потом соберите словарь `digit_embeddings` для первых 2000 цифр обучения,
сжатых `PCA(n_components=50, random_state=SEED)`: PCA с двумя компонентами,
`TSNE(n_components=2, random_state=SEED)` и `umap.UMAP(random_state=SEED)`,
ключи `"PCA"`, `"t-SNE"`, `"UMAP"`. Посчитайте `embedding_table` на сжатых цифрах.

In [ ]:
# X: (n, p), y: (n,), embeddings: {имя: (n, 2)}, n_neighbors >= 1
# возвращает таблицу (len(embeddings), 2): индекс — имена, столбцы "trustworthiness" и "accuracy kNN"
def embedding_scores(X: np.ndarray, y: np.ndarray, embeddings: dict[str, np.ndarray], n_neighbors: int = 10) -> pd.DataFrame:
    return ...


digits_50 = ...          # первые 2000 цифр обучения, сжатые до 50 компонент
digit_embeddings = ...   # {"PCA": (2000, 2), "t-SNE": (2000, 2), "UMAP": (2000, 2)}
embedding_table = ...

In [ ]:
# --- проверка ---
assert list(digit_embeddings) == ["PCA", "t-SNE", "UMAP"] and all(np.asarray(v).shape == (2000, 2) for v in digit_embeddings.values())
assert list(embedding_table.index) == ["PCA", "t-SNE", "UMAP"] and list(embedding_table.columns) == ["trustworthiness", "accuracy kNN"]
print(embedding_table.round(3))
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
for ax, (name, points) in zip(axes, digit_embeddings.items()):
    for digit_class in range(10):
        mask = yd_train[:2000] == digit_class
        ax.scatter(points[mask, 0], points[mask, 1], s=3, color=CLUSTER_COLORS[digit_class])
    ax.set_title(f"{name}: trustworthiness {embedding_table.loc[name, 'trustworthiness']:.2f}")
    ax.set_xticks([])
    ax.set_yticks([])
plt.tight_layout()
plt.show()
print("проверки пройдены")

**Вопрос.** Можно ли по картинке t-SNE сказать, что кластер нулей в два раза
больше кластера единиц или что тройки ближе к пятеркам, чем к нулям? Что
из увиденного на этих картинках можно утверждать, а что надо проверять
на исходных данных?

*Ответ пишите здесь.*